# F1-scientific-python — Session 04: Seaborn with Arrays

**Session length:** about 70 minutes • **Concept:** `seaborn-programming`

This session adds seaborn's statistical plotting layer without changing the data model you already know: every plotted variable is a NumPy array, and every figure is built on an explicit matplotlib `Axes`. Checkpoint answers are collected at the end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. The array-to-axes contract

Seaborn supplies high-level statistical marks; matplotlib still owns the figure and axes. Use this repeatable pattern:

1. create `fig, ax = plt.subplots(...)`;
2. pass NumPy arrays directly through named arguments such as `x=` and `y=`;
3. pass `ax=ax` so the destination is explicit;
4. label through `ax.set(...)`;
5. return or inspect `fig, ax`, then call `plt.show()` only at the presentation boundary.

Explicit axes make a plot testable: titles, labels, limits, legends, and drawn marks become ordinary object properties rather than visual guesses.

### Checkpoint 1

1. Why is `ax=ax` useful even when the notebook currently has only one figure?
2. Name three axes properties that code can check without comparing image pixels.

## 2. Histograms with fixed bins

A histogram is a counting rule. If two runs infer different bin edges, their bars are not directly comparable. For graded or comparative work, construct the edges yourself and pass them through `bins=`.

**Worked example 1 — a deterministic score histogram.** We choose half-point-offset edges spaced ten points apart, draw on an explicit axes, and verify the bar heights against `np.histogram`.

In [ ]:
scores = np.array([42, 55, 58, 61, 66, 68, 72, 72, 79, 84, 88, 91, 96])
edges = np.arange(39.5, 110.0, 10.0)

fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(x=scores, bins=edges, color="steelblue", edgecolor="black", ax=ax)
ax.set(title="Quiz-score distribution", xlabel="score", ylabel="count", xlim=(39.5, 109.5))

expected_counts, _ = np.histogram(scores, bins=edges)
drawn_counts = np.array([patch.get_height() for patch in ax.patches], dtype=int)
assert np.array_equal(drawn_counts, expected_counts)
assert ax.get_xlabel() == "score" and ax.get_ylabel() == "count"
plt.show()

The check compares meaning, not pixels: one height per declared bin, plus exact labels. Color-rendering differences between machines cannot break it.

### Checkpoint 2

1. Why is checking `len(ax.patches)` alone weaker than checking their heights?
2. For integer values 0 through 5 inclusive, give edges that center one bin on each integer.

## 3. Scatter plots, labels, and visual semantics

`sns.scatterplot` accepts separate arrays for horizontal position, vertical position, color grouping (`hue=`), and marker grouping (`style=`). A visual channel must encode something you can name. If both color and marker distinguish the same groups, the figure remains readable when printed without color.

**Worked example 2 — two cohorts on one axes.**

In [ ]:
hours = np.array([1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5])
scores = np.array([52, 60, 63, 70, 68, 78, 82, 91])
cohort = np.array(["morning", "evening", "morning", "evening",
                   "morning", "evening", "morning", "evening"])

fig, ax = plt.subplots(figsize=(6, 4))
sns.scatterplot(x=hours, y=scores, hue=cohort, style=cohort, s=80, ax=ax)
ax.set(title="Practice time and quiz score", xlabel="practice hours", ylabel="quiz score")
legend_text = [item.get_text() for item in ax.get_legend().get_texts()]
assert set(legend_text) == {"morning", "evening"}
assert ax.collections[0].get_offsets().shape == (8, 2)
plt.show()

`hue` and `style` are semantics: they map group values to visible properties and build a legend. `color="blue"` is only a fixed appearance; it carries no group meaning.

### Checkpoint 3

1. Explain the difference between `hue=cohort` and `color="blue"`.
2. Why can using `style=cohort` as well as `hue=cohort` improve accessibility?

## 4. Deterministic axes checks

A useful plot check targets the figure contract:

- exact title and axis-label strings;
- declared limits when the task pins them;
- number or values of histogram bars;
- number of plotted observations;
- legend labels representing the expected groups.

Avoid screenshot equality. Fonts, antialiasing, and library versions can change pixels without changing the statistical claim. For scatter points, collection offsets expose the plotted coordinate pairs.

In [ ]:
fig, ax = plt.subplots()
sns.scatterplot(x=np.array([0.0, 1.0, 2.0]), y=np.array([2.0, 1.0, 3.0]), ax=ax)
ax.set(title="Three observations", xlabel="input", ylabel="response")
offsets = ax.collections[0].get_offsets()
assert offsets.shape == (3, 2)
assert np.allclose(offsets[:, 0], np.array([0.0, 1.0, 2.0]), atol=1e-12, rtol=0)
assert ax.get_title() == "Three observations"
plt.show()

### Checkpoint 4

1. Why does a pixel-perfect image comparison test the rendering environment more than the plotting contract?
2. Which axes object exposes scatter coordinates, and what shape should its offsets have for 12 observations?

## 5. Reproducible comparisons

A fair two-group histogram needs more than two calls. Pin the random generator, create both arrays in a declared order, use the same bin edges, draw both on the same axes, label both, and use transparency so neither distribution hides the other. The computed summaries and the picture must describe the same arrays.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
baseline = rng.normal(loc=70.0, scale=8.0, size=200)
revised = rng.normal(loc=74.0, scale=8.0, size=200)
shared_edges = np.linspace(40.0, 100.0, 13)

fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(x=baseline, bins=shared_edges, alpha=0.45, label="baseline", ax=ax)
sns.histplot(x=revised, bins=shared_edges, alpha=0.45, label="revised", ax=ax)
ax.set(title="Two score distributions on shared bins", xlabel="score", ylabel="count")
ax.legend()
plt.show()

### Checkpoint 5

1. What goes wrong if each group silently chooses its own bin edges?
2. Why must the draw order from a seeded generator also stay fixed?

## 6. The justified matplotlib boundary

Use seaborn when a statistical mark and its semantics are the main job: distributions, grouped scatter, automatic legends. Use the axes API directly for simple geometric annotations or exact low-level control: reference lines, text, limits, and labels. Mixing them is intentional because seaborn draws onto a matplotlib axes.

For the comparison above, distribution bars belong to `sns.histplot`; an exact vertical mean marker is clearer as `ax.axvline`.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(x=revised, bins=shared_edges, color="teal", alpha=0.6, ax=ax)
ax.axvline(revised.mean(), color="black", linestyle="--", label="mean")
ax.set(title="Revised scores with mean marker", xlabel="score", ylabel="count")
ax.legend()
plt.show()

### Checkpoint 6

1. Justify `ax.axvline` rather than forcing the mean marker through a statistical plotting call.
2. Give one task where direct `ax.scatter` would be clearer than `sns.scatterplot`.

## 7. Common pitfalls and repair order

| Broken habit | Consequence | Repair |
|---|---|---|
| no `ax=` argument | marks may land on an unintended axes | create axes first and pass it explicitly |
| inferred bins for each group | bar heights answer different counting questions | construct one shared edge array |
| group color with no legend | readers cannot decode the groups | use a semantic mapping and inspect legend text |
| random samples with no seed | the claim changes on rerun | one `SEED`, one generator, fixed draw order |
| screenshot-only validation | harmless rendering changes look like failures | inspect axes objects and numerical summaries |
| high-level call for every annotation | simple geometry becomes indirect | use the axes API at a stated boundary |

A reliable repair order is data → bins/semantics → axes → labels → programmatic checks.

### Checkpoint 7

1. A two-group histogram reruns differently and has unlabeled colors. Name the first two repairs.
2. A plot looks right but `ax.get_ylabel()` is empty. Is the contract satisfied? Explain.

## 8. Exam connections and going deeper

Round 1 can grade plotting as a coding contract: required call, exact identifier, fixed bins, labels, and reproducible data. A visually attractive image that violates the named API or axes contract can still score zero. Practice p22 isolates histogram construction, p23 grades scatter semantics, and p24 integrates diagnosis, reproducibility, and the library boundary.

Later units use richer tabular workflows, but this session's main path stays deliberately array-oriented. The forward skill is not a new container; it is choosing statistical semantics deliberately and testing what the axes means.

### Checkpoint 8

1. Why can a beautiful plot still fail an exact-contract coding problem?
2. Which practice should you redo for (a) histogram bins, (b) scatter semantics, and (c) full comparison repair?

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. It pins the destination and makes later composition and testing unambiguous. 2. Examples: title, x/y labels, limits, bar heights, point offsets, legend text.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. The right number of bars can still have wrong counts. 2. `np.arange(-0.5, 6.5, 1.0)`.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. `hue` maps group values and creates semantics; fixed color only sets appearance. 2. Marker shape preserves the distinction without relying on color alone.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. Fonts and rendering can change while the statistical figure stays equivalent. 2. A collection's `get_offsets()` result, shape `(12, 2)`.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Corresponding bars represent different intervals. 2. One generator emits a sequence, so changing draw order changes which sample each group receives.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. A mean marker is exact geometric annotation, which the axes API expresses directly. 2. A fixed set of points needing no grouping or statistical legend.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. Pin the generator/draw order, then encode named groups with a legend. 2. No: the required y-label is missing even if a reader can guess it.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. Graders check the API and named artifacts, not only appearance. 2. p22, p23, and p24 respectively.

</details>